# Unemployment Rate Silver — approved annual release policy, guarded incremental MERGE

**Source:** `pondolens.dev_bronze.unemployment_rate`  
**Target:** `pondolens.dev_silver.unemployment_rate`

Project-approved policy: **2021 unverified exception (warning); 2022–2024 final; 2025 preliminary**. This is a project selection decision, not independent verification of official publication provenance. The notebook preserves all original Bronze rows, classifies each as accepted/rejected/excluded, and blocks the merge on conflicts, missing metadata, invalid values, reconciliation errors, duplicate keys, or snapshot mismatches.

**Important:** Execute cells in order, in one Databricks SQL session / job with overlapping runs disabled. Do not skip the pre-merge assertion. Release selections are keyed by *year and status*, not merely the title of a table. For 2021, the exception is explicitly documented. A 2021 region-year with multiple distinct rates still fails validation.


## 1. Create target and control tables (safe if they exist)


In [0]:
CREATE SCHEMA IF NOT EXISTS pondolens.dev_silver;
CREATE SCHEMA IF NOT EXISTS pondolens.dev_validation;
CREATE TABLE IF NOT EXISTS pondolens.dev_silver.unemployment_rate (
  unemployment_rate_id INT, year INT, region_code STRING, region_name STRING,
  unemployment_rate DECIMAL(12,4), run_id STRING, source STRING, processed_at TIMESTAMP
) USING DELTA;

-- Only verified source releases may compete for a region-year observation.
-- Populate manually from source documentation; never derive priority from rate/value.
CREATE TABLE IF NOT EXISTS pondolens.dev_validation.unemployment_release_registry (
  table_version STRING, estimate_status STRING, release_priority INT,
  is_verified BOOLEAN, evidence_reference STRING
) USING DELTA;

-- Explicitly approved complete-refresh years. Missing Silver keys in these years
-- are flagged, not deleted. Leave empty for incremental/partial batches.
CREATE TABLE IF NOT EXISTS pondolens.dev_validation.unemployment_snapshot_scope (
  year INT, is_complete BOOLEAN, evidence_reference STRING
) USING DELTA;


## 2. Inspect source releases and define project-approved policy

2022–2024: select `final`; 2025: select `preliminary`. For 2021, permit a release lacking verification under a documented exception. A verified source registry can be maintained separately, but these project approvals are **not** claims that official release documentation has been independently verified.

Review the distinct source labels before running the classification. No blanket approval for other years or estimate statuses. Snapshot scope is controlled separately and must have evidence for `is_complete = true`.


In [0]:
-- Source release labels, by year
SELECT TRY_CAST(TRY_CAST(TRIM(CAST(year AS STRING)) AS DECIMAL(10,1)) AS INT) AS clean_year,
       table_version, estimate_status, COUNT(*) AS rows
FROM pondolens.dev_bronze.unemployment_rate
WHERE LOWER(TRIM(geographic_level)) = 'region'
  AND LOWER(TRIM(indicator)) = 'unemployment rate'
GROUP BY 1, 2, 3 ORDER BY 1, 2, 3;

-- Project selection policy. This view intentionally does not label the source as independently verified.
CREATE OR REPLACE TEMP VIEW unemployment_release_policy AS
SELECT 2021 AS year, 'ANY' AS selected_status, 'APPROVED_UNVERIFIED_EXCEPTION' AS approval_type, 10 AS release_priority,
       '2021 source release unverified; accepted by project exception, subject to other quality checks' AS policy_note
UNION ALL SELECT 2022, 'final', 'PROJECT_APPROVED', 100, 'Final estimate selected by project policy'
UNION ALL SELECT 2023, 'final', 'PROJECT_APPROVED', 100, 'Final estimate selected by project policy'
UNION ALL SELECT 2024, 'final', 'PROJECT_APPROVED', 100, 'Final estimate selected by project policy'
UNION ALL SELECT 2025, 'preliminary', 'PROJECT_APPROVED', 100, 'Preliminary estimate selected by project policy';

SELECT * FROM unemployment_release_policy ORDER BY year;
SELECT * FROM pondolens.dev_validation.unemployment_snapshot_scope ORDER BY year;


## 3. Classify every raw row and standardize geography


In [0]:
CREATE OR REPLACE TEMP VIEW unemployment_raw AS
SELECT ROW_NUMBER() OVER (ORDER BY year, region, geographic_name, indicator,
  geographic_level, estimate_status, table_version, CAST(value AS STRING)) AS raw_row_id,
  b.*
FROM pondolens.dev_bronze.unemployment_rate b;

CREATE OR REPLACE TEMP VIEW unemployment_mapped AS
WITH normalized AS (
 SELECT b.*,
   TRY_CAST(TRY_CAST(TRIM(CAST(year AS STRING)) AS DECIMAL(10,1)) AS INT) AS clean_year,
   UPPER(TRIM(REGEXP_REPLACE(COALESCE(NULLIF(TRIM(region), ''), geographic_name), '\s+', ' '))) AS raw_region_name,
   TRY_CAST(REGEXP_REPLACE(TRIM(CAST(value AS STRING)), ',', '') AS DECIMAL(12,4)) AS clean_rate,
   LOWER(TRIM(estimate_status)) AS clean_status,
   TRIM(COALESCE(table_version, '')) AS clean_version,
   LOWER(TRIM(geographic_level)) AS clean_level,
   LOWER(TRIM(indicator)) AS clean_indicator
 FROM unemployment_raw b
), mapped AS (
 SELECT *,
   CASE
     WHEN raw_region_name RLIKE 'NATIONAL CAPITAL REGION|\\(NCR\\)' THEN 'NCR'
     WHEN raw_region_name RLIKE 'CORDILLERA ADMINISTRATIVE REGION|\\(CAR\\)' THEN 'CAR'
     WHEN raw_region_name RLIKE 'ILOCOS|REGION I \\(' THEN 'I'
     WHEN raw_region_name RLIKE 'CAGAYAN VALLEY|REGION II \\(' THEN 'II'
     WHEN raw_region_name RLIKE 'CENTRAL LUZON|REGION III \\(' THEN 'III'
     WHEN raw_region_name RLIKE 'CALABARZON|REGION IV-A|REGION IVA' THEN 'IVA'
     WHEN raw_region_name RLIKE 'MIMAROPA|REGION IV-B|REGION IVB' THEN 'IVB'
     WHEN raw_region_name RLIKE 'BICOL|REGION V \\(' THEN 'V'
     WHEN raw_region_name RLIKE 'WESTERN VISAYAS|REGION VI \\(' THEN 'VI'
     WHEN raw_region_name RLIKE 'CENTRAL VISAYAS|REGION VII \\(' THEN 'VII'
     WHEN raw_region_name RLIKE 'EASTERN VISAYAS|REGION VIII \\(' THEN 'VIII'
     WHEN raw_region_name RLIKE 'ZAMBOANGA PENINSULA|REGION IX \\(' THEN 'IX'
     WHEN raw_region_name RLIKE 'NORTHERN MINDANAO|REGION X \\(' THEN 'X'
     WHEN raw_region_name RLIKE 'DAVAO|REGION XI \\(' THEN 'XI'
     WHEN raw_region_name RLIKE 'SOCCSKSARGEN|REGION XII \\(' THEN 'XII'
     WHEN raw_region_name RLIKE 'CARAGA|REGION XIII \\(' THEN 'XIII'
     WHEN raw_region_name RLIKE 'BANGSAMORO|BARMM|ARMM|AUTONOMOUS REGION IN MUSLIM MINDANAO' THEN 'BARMM'
     WHEN raw_region_name RLIKE 'NEGROS ISLAND REGION|\\(NIR\\)' THEN 'NIR'
   END AS region_code
 FROM normalized
)
SELECT m.*,
 CASE region_code
 WHEN 'NCR' THEN 'National Capital Region' WHEN 'CAR' THEN 'Cordillera Administrative Region'
 WHEN 'I' THEN 'Ilocos Region' WHEN 'II' THEN 'Cagayan Valley' WHEN 'III' THEN 'Central Luzon'
 WHEN 'IVA' THEN 'CALABARZON' WHEN 'IVB' THEN 'MIMAROPA' WHEN 'V' THEN 'Bicol Region'
 WHEN 'VI' THEN 'Western Visayas' WHEN 'VII' THEN 'Central Visayas' WHEN 'VIII' THEN 'Eastern Visayas'
 WHEN 'IX' THEN 'Zamboanga Peninsula' WHEN 'X' THEN 'Northern Mindanao' WHEN 'XI' THEN 'Davao Region'
 WHEN 'XII' THEN 'SOCCSKSARGEN' WHEN 'XIII' THEN 'Caraga' WHEN 'BARMM' THEN 'BARMM'
 WHEN 'NIR' THEN 'Negros Island Region' END AS region_name,
 CASE WHEN clean_level <> 'region' OR clean_indicator <> 'unemployment rate' THEN 'OUT_OF_SCOPE'
      WHEN clean_year IS NULL OR clean_year NOT BETWEEN 2021 AND 2025 THEN 'INVALID_YEAR'
      WHEN region_code IS NULL THEN 'UNMAPPED_REGION'
      WHEN clean_rate IS NULL OR clean_rate NOT BETWEEN 0 AND 100 THEN 'INVALID_RATE'
      WHEN clean_version = '' OR clean_status IS NULL OR clean_status = '' THEN 'MISSING_RELEASE_METADATA'
      ELSE NULL END AS base_issue
FROM mapped m;


## 4. Apply the year-specific release policy and detect conflicts

Eligible releases are those matching the project policy. Preliminary rows superseded by a selected final release are **excluded**. Invalid eligible rows are **rejected**. All selected releases for a region-year must agree on rate; differing rates or multiple selected release identities cause a hard stop, rather than arbitrary selection. Identical repeated source rows are collapsed deterministically for the upsert, while still counted in input reconciliation.

`unverified_release` is an independent warning flag for 2021, not an additional row disposition. The `unemployment_release_registry` table remains available for independent provenance evidence but is not required for project-approved selection.


In [0]:
CREATE OR REPLACE TEMP VIEW unemployment_verified AS
SELECT m.*,
       p.release_priority, p.approval_type, p.policy_note,
       (m.clean_year = 2021) AS unverified_release,
       CASE WHEN m.clean_year = 2021 THEN
         '2021 data is unverified; accepted under documented project exception if all other checks pass'
         ELSE p.policy_note END AS verification_note,
       CASE
         WHEN m.base_issue IS NOT NULL THEN m.base_issue
         WHEN p.year IS NULL THEN 'UNVERIFIED_RELEASE'
         WHEN m.clean_year <> 2021 AND m.clean_status <> p.selected_status THEN 'SUPERSEDED_RELEASE'
         ELSE NULL END AS issue
FROM unemployment_mapped m
LEFT JOIN unemployment_release_policy p ON m.clean_year = p.year;

-- Conflicts within the same release identity (different values for same key).
CREATE OR REPLACE TEMP VIEW unemployment_conflicts AS
SELECT clean_year AS year, region_code, clean_version AS table_version,
       clean_status AS estimate_status, release_priority,
       COUNT(*) AS row_count, COUNT(DISTINCT clean_rate) AS distinct_rates,
       CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CAST(clean_rate AS STRING)))) AS conflicting_rates
FROM unemployment_verified
WHERE issue IS NULL
GROUP BY clean_year, region_code, clean_version, clean_status, release_priority
HAVING COUNT(DISTINCT clean_rate) > 1;

-- Different selected release identities for the same business key require explicit resolution.
CREATE OR REPLACE TEMP VIEW unemployment_priority_conflicts AS
SELECT clean_year AS year, region_code, release_priority,
       COUNT(DISTINCT clean_rate) AS distinct_rates,
       CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CONCAT(clean_version, ':', clean_status, ':', CAST(clean_rate AS STRING))))) AS revisions
FROM unemployment_verified
WHERE issue IS NULL
GROUP BY clean_year, region_code, release_priority
HAVING COUNT(DISTINCT clean_rate) > 1
    OR COUNT(DISTINCT CONCAT(clean_version, '|', clean_status)) > 1;

CREATE OR REPLACE TEMP VIEW unemployment_ranked AS
SELECT v.*,
       ROW_NUMBER() OVER (PARTITION BY clean_year, region_code
         ORDER BY clean_version DESC, clean_status DESC, raw_row_id) AS revision_rank
FROM unemployment_verified v
WHERE issue IS NULL;

CREATE OR REPLACE TEMP VIEW unemployment_classified AS
SELECT v.*,
 CASE WHEN v.base_issue = 'OUT_OF_SCOPE' THEN 'EXCLUDED'
      WHEN v.issue = 'SUPERSEDED_RELEASE' THEN 'EXCLUDED'
      WHEN v.issue IS NOT NULL THEN 'REJECTED'
      WHEN EXISTS (SELECT 1 FROM unemployment_conflicts c
         WHERE c.year = v.clean_year AND c.region_code = v.region_code)
        OR EXISTS (SELECT 1 FROM unemployment_priority_conflicts c
         WHERE c.year = v.clean_year AND c.region_code = v.region_code)
        THEN 'REJECTED'
      WHEN q.revision_rank = 1 THEN 'ACCEPTED'
      ELSE 'EXCLUDED' END AS row_disposition,
 CASE WHEN v.base_issue = 'OUT_OF_SCOPE' THEN 'OUT_OF_SCOPE'
      WHEN v.issue IS NOT NULL THEN v.issue
      WHEN EXISTS (SELECT 1 FROM unemployment_conflicts c
         WHERE c.year = v.clean_year AND c.region_code = v.region_code)
        THEN 'CONFLICTING_VALUES_SAME_RELEASE'
      WHEN EXISTS (SELECT 1 FROM unemployment_priority_conflicts c
         WHERE c.year = v.clean_year AND c.region_code = v.region_code)
        THEN 'CONFLICTING_VALUES_SELECTED_RELEASES'
      WHEN q.revision_rank > 1 THEN 'DUPLICATE_IDENTICAL_SELECTED_RELEASE'
      WHEN v.unverified_release THEN 'ACCEPTED_UNVERIFIED_2021_EXCEPTION'
      ELSE 'SELECTED_PROJECT_APPROVED_RELEASE' END AS disposition_reason
FROM unemployment_verified v
LEFT JOIN unemployment_ranked q ON v.raw_row_id = q.raw_row_id;

CREATE OR REPLACE TEMP VIEW unemployment_incoming AS
SELECT clean_year AS year, region_code, region_name, clean_rate AS unemployment_rate,
       'pondolens.dev_bronze.unemployment_rate' AS source
FROM unemployment_classified WHERE row_disposition = 'ACCEPTED';


## 5. Rejected-row output, unverified warnings, conflict detail and reconciliation

Inspect before proceeding. Exactly **input = accepted + rejected + excluded**. The unverified count is a subset of accepted/rejected/excluded and must not be added to the equation.


In [0]:
SELECT raw_row_id, year, region, geographic_name, value, table_version,
       estimate_status, region_code, unverified_release, verification_note, disposition_reason
FROM unemployment_classified WHERE row_disposition = 'REJECTED'
ORDER BY raw_row_id;

SELECT raw_row_id, clean_year, region_code, clean_version, clean_status,
       release_priority, clean_rate, unverified_release, verification_note,
       row_disposition, disposition_reason
FROM unemployment_classified WHERE unverified_release = TRUE
ORDER BY clean_year, region_code, raw_row_id;

SELECT raw_row_id, clean_year, region_code, clean_version, clean_status,
       release_priority, clean_rate, disposition_reason
FROM unemployment_classified
WHERE disposition_reason LIKE 'CONFLICTING_VALUES%'
ORDER BY clean_year, region_code, clean_version, raw_row_id;

SELECT * FROM unemployment_conflicts ORDER BY year, region_code;
SELECT * FROM unemployment_priority_conflicts ORDER BY year, region_code;

SELECT COUNT(*) AS input_rows,
 COUNT_IF(row_disposition = 'ACCEPTED') AS accepted_rows,
 COUNT_IF(row_disposition = 'REJECTED') AS rejected_rows,
 COUNT_IF(row_disposition = 'EXCLUDED') AS excluded_rows,
 COUNT_IF(unverified_release) AS unverified_release_rows,
 COUNT_IF(unverified_release AND row_disposition = 'ACCEPTED') AS accepted_unverified_rows,
 COUNT(*) - COUNT_IF(row_disposition IN ('ACCEPTED','REJECTED','EXCLUDED')) AS unclassified_rows,
 COUNT(*) - (COUNT_IF(row_disposition = 'ACCEPTED') + COUNT_IF(row_disposition = 'REJECTED') + COUNT_IF(row_disposition = 'EXCLUDED')) AS reconciliation_difference
FROM unemployment_classified;

SELECT clean_year, clean_status, row_disposition, disposition_reason,
       COUNT(*) AS rows
FROM unemployment_classified
GROUP BY clean_year, clean_status, row_disposition, disposition_reason
ORDER BY clean_year, clean_status, row_disposition, disposition_reason;


## 6. Snapshot reconciliation policy — flag removed records, never silently delete

Only `is_complete=true` years with evidence are treated as verified full-refresh scopes. Existing Silver region-year keys absent from the accepted input in those years are **flagged for investigation**. No automatic delete or deactivation is performed because the requested eight-column Silver schema has no removal-status column. For partial/incremental batches, missing records are **not** interpreted as removed. The preflight below blocks MERGE on any verified-scope missing keys, until the source is corrected or an explicit separate removal policy is approved.


In [0]:
CREATE OR REPLACE TEMP VIEW unemployment_snapshot_missing AS
SELECT t.year, t.region_code, t.region_name, t.unemployment_rate_id,
       'MISSING_FROM_VERIFIED_COMPLETE_SNAPSHOT' AS flag
FROM pondolens.dev_silver.unemployment_rate t
JOIN pondolens.dev_validation.unemployment_snapshot_scope s
 ON t.year = s.year AND s.is_complete = TRUE
LEFT ANTI JOIN unemployment_incoming i
 ON t.year = i.year AND t.region_code = i.region_code;

SELECT * FROM unemployment_snapshot_missing ORDER BY year, region_code;

-- Scope must have nonblank documentary evidence; duplicate scopes are invalid.
SELECT year, COUNT(*) AS n
FROM pondolens.dev_validation.unemployment_snapshot_scope
GROUP BY year HAVING COUNT(*) > 1;


## 7. Assign new IDs only to new business keys; preserve existing IDs


In [0]:
CREATE OR REPLACE TEMP VIEW unemployment_upsert AS
WITH current_max AS (
 SELECT GREATEST(COALESCE(MAX(CAST(unemployment_rate_id AS BIGINT)), 1000000), 1000000) AS max_id
 FROM pondolens.dev_silver.unemployment_rate
), new_keys AS (
 SELECT i.year, i.region_code, ROW_NUMBER() OVER (ORDER BY i.year, i.region_code) AS seq
 FROM unemployment_incoming i
 LEFT ANTI JOIN pondolens.dev_silver.unemployment_rate t
 ON i.year = t.year AND i.region_code = t.region_code
), numbered AS (
 SELECT k.year, k.region_code, m.max_id + k.seq AS new_id
 FROM new_keys k CROSS JOIN current_max m
)
SELECT i.*, n.new_id AS new_id_bigint
FROM unemployment_incoming i LEFT JOIN numbered n
ON i.year = n.year AND i.region_code = n.region_code;

SELECT COUNT(*) AS incoming_keys, COUNT(new_id_bigint) AS new_keys,
 MAX(new_id_bigint) AS highest_proposed_id FROM unemployment_upsert;


## 8. HARD STOP — validate before the MERGE

Run this cell after reviewing the rejected and conflict outputs. Any failure raises an error and must stop the job. Do not run the MERGE cell manually after a failed assertion. No Silver data is modified by the preceding view/check cells. This check also validates registry uniqueness, verified snapshot metadata, target key/ID integrity and INT capacity.


In [0]:
SELECT assert_true(
  (SELECT COUNT(*) FROM unemployment_classified) > 0
  AND (SELECT COUNT(*) FROM unemployment_incoming) > 0
  AND (SELECT COUNT(*) FROM unemployment_classified WHERE row_disposition = 'REJECTED') = 0
  AND (SELECT COUNT(*) FROM unemployment_conflicts) = 0
  AND (SELECT COUNT(*) FROM unemployment_priority_conflicts) = 0
  AND (SELECT COUNT(*) FROM unemployment_classified) =
      (SELECT COUNT(*) FROM unemployment_classified WHERE row_disposition IN ('ACCEPTED','REJECTED','EXCLUDED'))
  AND (SELECT COUNT(*) FROM pondolens.dev_validation.unemployment_snapshot_scope
       WHERE is_complete = TRUE AND NULLIF(TRIM(evidence_reference),'') IS NULL) = 0
  AND (SELECT COUNT(*) FROM (SELECT year FROM pondolens.dev_validation.unemployment_snapshot_scope
       GROUP BY year HAVING COUNT(*) > 1)) = 0
  AND (SELECT COUNT(*) FROM unemployment_snapshot_missing) = 0
  AND (SELECT COUNT(*) FROM (SELECT year, region_code FROM unemployment_incoming
       GROUP BY year, region_code HAVING COUNT(*) > 1)) = 0
  AND (SELECT COUNT(*) FROM pondolens.dev_silver.unemployment_rate
       WHERE year IS NULL OR region_code IS NULL OR unemployment_rate_id IS NULL OR unemployment_rate_id < 1000001) = 0
  AND (SELECT COUNT(*) FROM (SELECT year, region_code FROM pondolens.dev_silver.unemployment_rate
       GROUP BY year, region_code HAVING COUNT(*) > 1)) = 0
  AND (SELECT COUNT(*) FROM (SELECT unemployment_rate_id FROM pondolens.dev_silver.unemployment_rate
       GROUP BY unemployment_rate_id HAVING COUNT(*) > 1)) = 0
  AND (SELECT COALESCE(MAX(new_id_bigint),0) FROM unemployment_upsert) <= 2147483647,
  'STOP: rejected rows, selected-release conflicts, bad reconciliation/snapshot scope, duplicate keys, or ID overflow. Review diagnostics; fix source/policy; rerun.'
) AS premerge_validation;


## 9. Incremental MERGE — run only after successful preflight

Updates changed rate/name/source while retaining the existing integer ID; inserts only unseen business keys. Repeated runs with unchanged data do not update records. The staging views must be regenerated and preflight rerun for every batch.


In [0]:
MERGE INTO pondolens.dev_silver.unemployment_rate AS t
USING unemployment_upsert AS s
ON t.year = s.year AND t.region_code = s.region_code
WHEN MATCHED AND (
 NOT (t.unemployment_rate <=> s.unemployment_rate)
 OR NOT (t.region_name <=> s.region_name)
 OR NOT (t.source <=> s.source)
) THEN UPDATE SET
 t.region_name = s.region_name,
 t.unemployment_rate = s.unemployment_rate,
 t.source = s.source,
 t.run_id = CONCAT('unemployment_', DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMdd_HHmmss')),
 t.processed_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (
 unemployment_rate_id, year, region_code, region_name,
 unemployment_rate, run_id, source, processed_at
) VALUES (
 CAST(s.new_id_bigint AS INT), s.year, s.region_code, s.region_name,
 s.unemployment_rate,
 CONCAT('unemployment_', DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMdd_HHmmss')),
 s.source, CURRENT_TIMESTAMP()
);


## 10. Post-merge validation and audit


In [0]:
SELECT COUNT(*) AS silver_rows, COUNT(DISTINCT CONCAT(CAST(year AS STRING),'|',region_code)) AS distinct_keys,
 MIN(unemployment_rate_id) AS min_id, MAX(unemployment_rate_id) AS max_id
FROM pondolens.dev_silver.unemployment_rate;
SELECT year, COUNT(*) AS region_count FROM pondolens.dev_silver.unemployment_rate GROUP BY year ORDER BY year;
SELECT i.year, i.region_code, i.unemployment_rate AS expected_rate,
       t.unemployment_rate AS saved_rate
FROM unemployment_incoming i
LEFT JOIN pondolens.dev_silver.unemployment_rate t
 ON i.year = t.year AND i.region_code = t.region_code
WHERE t.unemployment_rate_id IS NULL OR NOT (i.unemployment_rate <=> t.unemployment_rate)
   OR NOT (i.region_name <=> t.region_name);


### Operational notes

- **2021 is not verified.** Its records are accepted only under a documented project exception, and must still pass value, region, key and conflict checks.
- **2022–2024 final, 2025 preliminary** are project-approved selection rules. Source authenticity and finality should be corroborated with release evidence before claiming independent verification.
- **Input reconciliation:** all Bronze rows are classified; excluded superseded releases are still counted. Accepted rows are the single selected record per business key.
- **Snapshot scope:** only years explicitly marked `is_complete = TRUE` in `pondolens.dev_validation.unemployment_snapshot_scope` are tested for removals. Missing keys block merge and are never deleted automatically. Leave scope empty for partial batches.
- **ID behavior:** existing IDs are preserved; new IDs start at 1000001 and increment from current max. Overflow is checked before MERGE. Existing target data is not modified when the pre-merge assertion fails.
- **Single writer:** disable concurrent runs. Temporary views depend on the current SQL session. No persistent rejected-row audit table is created; export diagnostic results if retention is required.
- **Source row ID:** `raw_row_id` is run-local and is not a permanent Bronze identifier.
- **Re-running:** changes to selected source data are upserted by `(year, region_code)`. Unchanged target records remain untouched.
